# Stage 1b — E6 cross-mechanism + E7 drive-swap  (heat = TFF)

Runs the two Stage-1b campaigns on **WRN-28-10, CIFAR-10-C severity 5, seed 42**,
single-corruption streams, batch 64 — the same runner/pipeline as E1.

**Criterion (Stage-1b):** the **primary** law criterion is **hard collapse**
(`nan_inf` / chance), matching the Stage-1 forward confirmation.
`soft:below_source` is recorded per run and reported as a **separate** boundary,
never mixed into law fits. All slope comparisons use the A2 bracket-weighted
hard-only Bernoulli reference **`S_frozen = 1.185`** (default; recorded in every
predictions file).

**Both campaigns are pre-registered.** Each writes its predictions
(`e6_predictions.json` / `e7_predictions.json`, timestamped) **before** any
grid run; the file is never overwritten and pre-existing non-ladder grid runs
are recorded as violations.

- **E6** (`HeatAnchor`, ~25–35 runs): θ ← θ − ηλ(θ − θ_src) after each SGD step,
  mutually exclusive with the Bernoulli restore. Cells:
  `{gaussian_noise × 5e-4,1e-3,2e-3} + {elastic_transform × 1e-3}`.
  Verdict: **SAME-CURVE** iff anchor slope within ±25% of `S_frozen` **and**
  pooled R² ≥ 0.95 (anchor + Bernoulli hard points) **and** ≥ 3/4 cells within
  ±25% of the frozen prediction (or in-bracket); else **DIFFERENT-CURVE**.
- **E7** (`HeatEntropyDrive`, ~18–26 runs): loss = mean prediction entropy of
  the batch. Cells: `{gaussian_noise, elastic_transform} × {5e-4,1e-3,2e-3}`.
  Verdict: **DRIVE-AGNOSTIC-STRONG / -WEAK / DRIVE-SENSITIVE / UNCLASSIFIABLE**,
  plus the calmness comparison at matched stable cells.

Both are **resumable** (re-run a cell after a disconnect). Open in Colab, set
**Runtime → Change runtime type → GPU (L4/A100)**, run top-to-bottom.
Rough cost: ~1–1.5 h on A100 for both (a single-corruption run ≈ 1/15 of a
continual run).

**STOP after E6 + E7; Stage 2 awaits instruction.**

## 1. GPU check

In [ ]:
# A GPU is strongly recommended.
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode == 0:
    print(out.stdout)
else:
    print("WARNING: No GPU detected. Set Runtime -> Change runtime type -> GPU (L4/A100).")
    print("         Stage 1b will still run on CPU but far slower than the quoted times.")

## 2. Config — `# === EDIT ME ===`

In [ ]:
# === EDIT ME ===========================================================
# --- Repo ---
REPO_URL   = "https://github.com/octadion/heat.git"   # or upload the repo manually
REPO_DIR   = "heat"                                    # folder name after clone
GIT_BRANCH = ""                                        # "" = default branch

# --- Where results live (Drive-backed so they survive a disconnect) ---
# IMPORTANT: point this at the SAME dir that holds your E1 run JSONs, so E6/E7
# reuse the E1 source runs and the Bernoulli hard points for the pooled fit.
USE_DRIVE         = True
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/pstar_results"
LOCAL_RESULTS_DIR = "/content/pstar_results"           # used only if USE_DRIVE = False

# --- Source checkpoint (WRN-28-10 — the E1 artifact) ---
# May be: a direct http(s) URL (wget), a Google-Drive file id or share URL
# (gdown), OR a local path already on disk. Leave "" to TRAIN from scratch
# (requires TRAIN_IF_MISSING = True). Normalized to
# experiments/checkpoints/wrn28_10_final.pt.
CKPT_DIR         = "/content/drive/MyDrive/heat/experiments/checkpoints"
CKPT_WRN         = CKPT_DIR + "/wrn28_10_final.pt"
TRAIN_IF_MISSING = False
TRAIN_EPOCHS     = 30

# --- Stage-1b reference slope ---
# Bracket-weighted hard-only Bernoulli slope (A2), fixed by the GO instruction.
# Cell 8 can RE-FREEZE it from your E1 JSONs (audit); set REFREEZE_FROM_E1 = True
# to use that value instead of this literal.
S_FROZEN         = 1.185
REFREEZE_FROM_E1 = False

# --- Which campaigns to run ---
RUN_E6 = True
RUN_E7 = True

# --- RULE 3 (pre-registration validity; see README_stage1.md) ---
# If an e6/e7_predictions.json already exists it is INVALID by default and the
# campaign ABORTS. Choose explicitly:
#   "abort"        -> refuse to proceed (default; forces a decision)
#   "trust"        -> reuse it (ONLY if scripts/audit_stage1b.py confirmed its
#                     reference ||g_bar|| runs were genuine)
#   "requarantine" -> quarantine it to analysis/quarantine/ and re-freeze new
#                     predictions BEFORE any new grid run
PREDICTIONS_POLICY = "abort"
# RULE 2: set True ONLY to deliberately run without the E1 reference files in
# RESULTS_DIR (the verdict will be VOID without them).
FRESH_REFERENCE = False

# --- Grid / search (defaults match the master plan; rarely edited) ---
P_GRID       = [0.0, 0.005, 0.010, 0.020]   # BASE coarse grid at eta=1e-3;
                                            # auto-scaled by eta/1e-3 per cell
BISECT_STEPS = 3
SEED         = 42

# --- Data / loader ---
C10C_ROOT   = "data/cifar10c"    # relative to the repo dir (we cd into it)
BATCH_SIZE  = 64
NUM_WORKERS = 2
# =======================================================================

RESULTS_DIR = DRIVE_RESULTS_DIR if USE_DRIVE else LOCAL_RESULTS_DIR
RULE_FLAGS = []
if PREDICTIONS_POLICY == "trust":
    RULE_FLAGS.append("--trust-existing-predictions")
elif PREDICTIONS_POLICY == "requarantine":
    RULE_FLAGS.append("--requarantine-predictions")
if FRESH_REFERENCE:
    RULE_FLAGS.append("--fresh-reference")
print("RESULTS_DIR =", RESULTS_DIR)
print("S_FROZEN    =", S_FROZEN, "(refreeze from E1:", REFREEZE_FROM_E1, ")")
print("run E6:", RUN_E6, "| run E7:", RUN_E7, "| rule flags:", RULE_FLAGS)

## 3. Mount Google Drive (if `USE_DRIVE`)

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results dir:", RESULTS_DIR)
print("(Re-running a campaign cell after a disconnect resumes from here.)")

## 4. Clone the repo and install dependencies

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    cmd = ["git", "clone"]
    if GIT_BRANCH:
        cmd += ["--branch", GIT_BRANCH]
    cmd += [REPO_URL, REPO_DIR]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f"[skip clone] {REPO_DIR}/ already exists.")

os.chdir("/content/" + REPO_DIR if not os.path.isabs(REPO_DIR) else REPO_DIR)
REPO_ROOT = os.getcwd()
print("cwd =", REPO_ROOT)

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run(["pip", "install", "-q", "gdown"], check=False)

os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
# run_tier2 prints a U+2192 in its summary; force UTF-8 in child processes so a
# non-UTF-8 locale can't crash a run before its JSON is saved (no-op on Colab).
os.environ["PYTHONUTF8"] = "1"
print("PYTHONPATH =", os.environ["PYTHONPATH"])

## 5. Get CIFAR-10-C (all 5 severities)

In [ ]:
import subprocess, numpy as np, os
subprocess.run(["python", "scripts/download_cifar10c.py", "--root", C10C_ROOT], check=True)

labels_path = os.path.join(C10C_ROOT, "labels.npy")
assert os.path.exists(labels_path), f"labels.npy missing in {C10C_ROOT}"
n_labels = len(np.load(labels_path))
assert n_labels == 50000, f"expected 50000 labels (5 severities), got {n_labels}"
probe = os.path.join(C10C_ROOT, "gaussian_noise.npy")
n_imgs = np.load(probe, mmap_mode="r").shape[0]
assert n_imgs == 50000, f"gaussian_noise.npy has {n_imgs} rows, expected 50000"
print(f"[ok] CIFAR-10-C present with all 5 severities ({n_imgs} imgs/corruption).")

## 6. Get the WRN-28-10 source checkpoint

In [ ]:
import os, subprocess

def _looks_like_url(s):
    return s.startswith("http://") or s.startswith("https://")

def resolve_ckpt(arch, spec):
    dest = f"experiments/checkpoints/{arch}_final.pt"
    os.makedirs("experiments/checkpoints", exist_ok=True)
    if os.path.exists(dest):
        print(f"[skip] {arch}: {dest} already present.")
        return dest
    if spec:
        if os.path.exists(spec):
            print(f"[local] {arch}: using {spec}")
            return spec
        if (os.path.isabs(spec) or os.sep in spec) and not _looks_like_url(spec):
            raise SystemExit(
                f"[fatal] {arch}: local checkpoint not found: {spec}\n"
                f"        Check the path or set TRAIN_IF_MISSING = True.")
        if _looks_like_url(spec) and "drive.google.com" not in spec:
            print(f"[wget] {arch}: {spec}")
            subprocess.run(["wget", "-q", "-O", dest, spec], check=True)
        else:
            print(f"[gdown] {arch}: {spec}")
            if _looks_like_url(spec):
                subprocess.run(["gdown", "--fuzzy", "-O", dest, spec], check=True)
            else:
                subprocess.run(["gdown", "--id", spec, "-O", dest], check=True)
        return dest
    if TRAIN_IF_MISSING:
        print(f"[train] {arch}: training source model ({TRAIN_EPOCHS} epochs)")
        subprocess.run(["python", "scripts/train_source.py", "--arch", arch,
                        "--epochs", str(TRAIN_EPOCHS)], check=True)
        return dest
    raise SystemExit(f"[fatal] no checkpoint for {arch}. Set CKPT_WRN or "
                     f"TRAIN_IF_MISSING = True.")

CKPT_WRN_PATH = resolve_ckpt("wrn28_10", CKPT_WRN)
print("\nWRN checkpoint ->", CKPT_WRN_PATH, "(exists=%s)" % os.path.exists(CKPT_WRN_PATH))

## 7. Pipeline validation (mandatory gate)

Reproduces the known WRN sev-5 collapse anchor (`p=0 → NaN`, `p=0.005 → later
NaN`, `p≥0.010 → stable`, ‖ḡ‖ ∈ [8,18]). Reuses the WRN sev5 JSONs already on
Drive, so it is essentially free — **do not skip it**.

In [ ]:
import subprocess
rc = subprocess.run([
    "python", "scripts/run_pstar_sweep.py", "--sanity-check-only",
    "--results-dir", RESULTS_DIR, "--ckpt-wrn", CKPT_WRN_PATH,
    "--c10c-root", C10C_ROOT, "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
]).returncode
assert rc == 0, ("PIPELINE VALIDATION FAILED (see output). Do NOT run the "
                 "campaigns. Check the WRN checkpoint, the 5 severities, GPU.")
print("\nPipeline validated. Stage-1b campaigns are safe to run.")

## 8. (Optional) Re-freeze `S_frozen` from your E1 JSONs — audit

The GO instruction fixes `S_frozen = 1.185`. If your E1 run JSONs are in
`RESULTS_DIR`, this cell recomputes the A2 bracket-weighted hard-only pooled
Bernoulli slope and (only if `REFREEZE_FROM_E1 = True`) uses that value for E6/E7
instead of the literal. It also writes `analysis/stage1_softboundary.md` and
`analysis/stage1_hardslope_frozen.json`. Safe to skip — E6/E7 take `--s-frozen`
directly.

In [ ]:
import subprocess, os, json
rc = subprocess.run(["python", "scripts/analyze_stage1_followup.py",
                     "--results-dir", RESULTS_DIR, "--seed", str(SEED)]).returncode
frozen_path = os.path.join(RESULTS_DIR, "analysis", "stage1_hardslope_frozen.json")
if rc == 0 and os.path.exists(frozen_path):
    S = json.load(open(frozen_path))["S_frozen"]
    print(f"\n[A2] recomputed pooled bracket-weighted hard-only slope = {S}")
    if REFREEZE_FROM_E1 and S:
        S_FROZEN = float(S)
        print(f"[A2] REFREEZE_FROM_E1 = True -> using S_FROZEN = {S_FROZEN}")
    else:
        print(f"[A2] keeping the GO literal S_FROZEN = {S_FROZEN} "
              f"(set REFREEZE_FROM_E1 = True to use the recomputed value).")
else:
    print("[A2] skipped/failed (E1 JSONs not found in RESULTS_DIR). "
          f"Proceeding with S_FROZEN = {S_FROZEN}.")

## 8b. Forensic audit (Part A — zero GPU, read-only)

Run this BEFORE any (re-)campaign when verdicts look inconsistent, and after
the campaigns as a sanity pass. Prints one-line root causes (A1 paths/filters,
A2/A3 trajectory forensics incl. the lambda=0 vs E1 p=0 side-by-side, A4
dispatch-from-recorded-args) and writes `analysis/stage1b_audit.md`.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/audit_stage1b.py",
                "--results-dir", RESULTS_DIR,
                "--e1-results-dir", RESULTS_DIR,
                "--cell", "gaussian_noise:2e-3"])

## 9. E6 — cross-mechanism campaign (resumable)

Runs `run_stage1b_e6.py`: phase 1 (source + reference-λ for all cells) → writes
`e6_predictions.json` → phase 3 (at-λ̂ run, λ grid, upward extension, bisection,
hard criterion). Re-run this cell after a disconnect to continue.

In [ ]:
import subprocess
if RUN_E6:
    rc = subprocess.run([
        "python", "scripts/run_stage1b_e6.py",
        "--results-dir", RESULTS_DIR, "--ckpt-wrn", CKPT_WRN_PATH,
        "--c10c-root", C10C_ROOT, "--s-frozen", str(S_FROZEN),
        "--p-grid", *[str(p) for p in P_GRID],
        "--bisect-steps", str(BISECT_STEPS), "--seed", str(SEED),
        "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
        *RULE_FLAGS,
    ]).returncode
    print("\n[E6] exit code", rc)
    if rc != 0:
        print("[E6] ABORTED — read the message above (RULE 2/3): set "
              "PREDICTIONS_POLICY / FRESH_REFERENCE in the config cell "
              "deliberately, then re-run.")
else:
    print("[E6] skipped (RUN_E6 = False).")

### E6 verdict + plot (inline)

In [ ]:
import os
from IPython.display import Image, Markdown, display
adir = os.path.join(RESULTS_DIR, "analysis")
md_path = os.path.join(adir, "e6_crossmech.md")
png_path = os.path.join(adir, "e6_crossmech.png")
if os.path.exists(md_path):
    display(Markdown(open(md_path, encoding="utf-8").read()))
else:
    print("e6_crossmech.md not found — run cell 9, or re-display with "
          "`python scripts/run_stage1b_e6.py --results-dir <dir> --analyze-only`.")
if os.path.exists(png_path):
    display(Image(filename=png_path))

## 10. E7 — drive-swap campaign (resumable)

Runs `run_stage1b_e7.py`: entropy-drive reference runs → `e7_predictions.json`
(strong-form p̂) → at-p̂ run, p grid, bisection (hard), plus the calmness
comparison at matched stable cells.

In [ ]:
import subprocess
if RUN_E7:
    rc = subprocess.run([
        "python", "scripts/run_stage1b_e7.py",
        "--results-dir", RESULTS_DIR, "--ckpt-wrn", CKPT_WRN_PATH,
        "--c10c-root", C10C_ROOT, "--s-frozen", str(S_FROZEN),
        "--p-grid", *[str(p) for p in P_GRID],
        "--bisect-steps", str(BISECT_STEPS), "--seed", str(SEED),
        "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
        *RULE_FLAGS,
    ]).returncode
    print("\n[E7] exit code", rc)
    if rc != 0:
        print("[E7] ABORTED — read the message above (RULE 2/3): set "
              "PREDICTIONS_POLICY / FRESH_REFERENCE in the config cell "
              "deliberately, then re-run.")
else:
    print("[E7] skipped (RUN_E7 = False).")

### E7 verdict + plot (inline)

In [ ]:
import os
from IPython.display import Image, Markdown, display
adir = os.path.join(RESULTS_DIR, "analysis")
md_path = os.path.join(adir, "e7_driveswap.md")
png_path = os.path.join(adir, "e7_driveswap.png")
if os.path.exists(md_path):
    display(Markdown(open(md_path, encoding="utf-8").read()))
else:
    print("e7_driveswap.md not found — run cell 10, or re-display with "
          "`python scripts/run_stage1b_e7.py --results-dir <dir> --analyze-only`.")
if os.path.exists(png_path):
    display(Image(filename=png_path))

## 11. Done — STOP

Stage 1b artifacts are in `RESULTS_DIR/analysis/`:
`e6_predictions.json`, `e6_crossmech.{json,png,md}`, `e7_predictions.json`,
`e7_driveswap.{json,png,md}`, plus `stage1b_e6_manifest.jsonl` /
`stage1b_e7_manifest.jsonl`.

**STOP — Stage 1b complete (E6 + E7). Stage 2 awaits instruction.**